# 🔢 Módulo 03 · NumPy

Cálculo numérico vectorizado desde el punto de vista práctico: cuándo usar arrays, cómo leer su forma y cómo resumir datos por filas o columnas.

Versión web con conceptos, reconocimiento, razonamiento y proyecto: [https://rosalesluciano.github.io/datalab/](https://rosalesluciano.github.io/datalab/#/m/m03)

## Cómo trabajar este notebook
1. Ejecutá la celda ⚙️ (carga datos y verificador).
2. **Antes de escribir código**, escribí en un comentario cómo lo vas a resolver.
3. Implementá y guardá tu respuesta en `resultado` (o lo que pida el ejercicio).
4. `comprobar()` verifica. Si te trabás: `ver_pista("id", 1)`, después 2 y 3.
5. `ver_solucion("id")` es el último recurso: muestra idea → pseudocódigo → código.

## Conceptos del módulo
- **¿Por qué NumPy y no Python puro?**: Hacer cálculos numéricos sobre **muchos valores a la vez**, rápido y con código corto. NumPy es el motor numérico debajo de pandas, matplotlib y scikit-learn.
- **Arrays: shape, dimensiones y dtype**: Crear arrays y entender su **forma** (cuántas filas/columnas/dimensiones) y su **tipo** (todos los elementos comparten uno).
- **Indexar, recortar y filtrar**: Extraer partes de un array: un elemento, una fila, una columna, un bloque o los elementos que cumplen una condición.
- **Operaciones vectorizadas**: Aplicar fórmulas a todos los elementos de una vez: conversiones, normalizaciones, impuestos.
- **Broadcasting**: Combinar arrays de distinta forma sin copiar datos: sumar un bonus por examen a toda la matriz, restar la media de cada columna.
- **Estadística y ejes**: Resumir datos numéricos: promedios, dispersión, percentiles, por fila o por columna.
- **Condicionales vectorizados**: Elegir un valor u otro según una condición, para todos los elementos a la vez.
- **Aleatoriedad reproducible**: Generar datos aleatorios (simulaciones, muestras, datos de prueba) que se puedan **reproducir**.
- **NumPy y pandas**: Entender que cada columna de pandas guarda sus datos en un array de NumPy, y pasar de uno a otro cuando hace falta.

In [ ]:
#@title ⚙️ 1) Ejecutá esta celda primero (datos y verificador) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series", list: "una lista", dict: "un diccionario",
               tuple: "una tupla", set: "un set", str: "un texto (str)", int: "un entero (int)",
               float: "un decimal (float)", bool: "un booleano (bool)", type(None): "None (nada)",
               np.ndarray: "un array de NumPy"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    if isinstance(e, dict):
        return (isinstance(u, dict) and set(map(str, u)) == set(map(str, e))
                and all(_scalar_eq(u[k] if k in u else u[str(k)], v) for k, v in e.items()))
    if isinstance(e, (set, frozenset)):
        return isinstance(u, (set, frozenset)) and u == e
    try:
        return bool(u == e)
    except Exception:
        return False


def _array_compare(u, e):
    if isinstance(u, (pd.DataFrame, pd.Series)):
        u = u.to_numpy()
    try:
        ua = np.asarray(u)
    except Exception:
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if not isinstance(u, np.ndarray) and not isinstance(u, (list, tuple)):
        return False, f"Se esperaba un array de NumPy y tu resultado es {_tipo(u)}."
    if ua.shape != e.shape:
        return False, f"Forma distinta: tu array tiene shape {ua.shape} y se esperaba {e.shape}."
    num = np.issubdtype(e.dtype, np.number) or e.dtype == bool
    try:
        if num and (np.issubdtype(ua.dtype, np.number) or ua.dtype == bool):
            ok = np.allclose(ua.astype(float), e.astype(float), rtol=1e-6, atol=1e-6, equal_nan=True)
        else:
            ok = ua.tolist() == e.tolist()
    except Exception:
        ok = False
    return (True, "") if ok else (False, "La forma es correcta pero los valores no coinciden. Revisa el cálculo.")


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, np.ndarray):
        return _array_compare(u, e)
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    if u is None:
        return False, ("Tu resultado es None. ¿Te falta un `return` en la función, o guardaste lo que devuelve "
                       "un método que modifica en el lugar (como `.sort()` o `.append()`)?")
    contenedores = (list, tuple, dict, set, str)
    if isinstance(e, contenedores) and not isinstance(u, type(e)) and not (
            isinstance(e, (list, tuple)) and isinstance(u, (list, tuple))):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}."
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple)) and len(u) != len(e):
        return False, f"Tu resultado tiene {len(u)} elementos y se esperaban {len(e)}."
    if isinstance(e, dict) and isinstance(u, dict) and set(map(str, u)) != set(map(str, e)):
        faltan = [k for k in e if str(k) not in set(map(str, u))]
        sobran = [k for k in u if str(k) not in set(map(str, e))]
        msg = "Las claves del diccionario no coinciden."
        if faltan:
            msg += f" Faltan: {faltan}."
        if sobran:
            msg += f" Sobran: {sobran}."
        return False, msg
    txt = repr(u)
    return False, f"Tu resultado es {txt[:300] + ('…' if len(txt) > 300 else '')}, pero no es el valor esperado."


# ---------------------------------------------------------------- gráficos
def _plt():
    import matplotlib.pyplot as plt
    return plt


def _fig():
    plt = _plt()
    return plt.gcf() if plt.get_fignums() else None


def _ax(i=0):
    f = _fig()
    if f is None or len(f.axes) <= i:
        raise LookupError("no hay ningún gráfico dibujado")
    return f.axes[i]


def _legend(ax=None):
    ax = ax or _ax()
    lg = ax.get_legend()
    return [t.get_text() for t in lg.get_texts()] if lg else []


def _heights(ax=None):
    return [round(float(p.get_height()), 6) for p in (ax or _ax()).patches]


def _widths(ax=None):
    return [round(float(p.get_width()), 6) for p in (ax or _ax()).patches]


def _xticks(ax=None):
    return [t.get_text() for t in (ax or _ax()).get_xticklabels()]


def _hex(color):
    import matplotlib.colors as mcolors
    return mcolors.to_hex(color)


_HELPERS = {"_fig": _fig, "_ax": _ax, "_legend": _legend, "_heights": _heights,
            "_widths": _widths, "_xticks": _xticks, "_hex": _hex}


def pq_prepare():
    """Deja matplotlib limpio antes de ejecutar código (si está cargado)."""
    import sys
    if "matplotlib.pyplot" in sys.modules:
        plt = sys.modules["matplotlib.pyplot"]
        plt.close("all")
        plt.show = lambda *a, **k: None


def pq_check(ns, get_expected, opts):
    """Evalúa el ejercicio.

    ns = namespace del alumno. get_expected = función que devuelve el namespace de la
    solución (solo se ejecuta si hace falta, para no pisar los gráficos del alumno).
    """
    custom = opts.get("custom")
    if custom:
        if opts.get("plot") and _fig() is None:
            return False, "Todavía no hay ningún gráfico. ¡Dibuja algo! 🎨"
        env = dict(_HELPERS)
        env.update(ns)
        try:
            ok = bool(eval(custom, env))
        except Exception as err:
            return False, f"{opts.get('custom_msg', 'Todavía no es correcto.')} ({type(err).__name__}: {err})"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    exp_ns = get_expected() if callable(get_expected) else get_expected
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usá & (y), | (o) y poné cada condición entre paréntesis."),
    ("can't multiply sequence", "Estás multiplicando un texto (str). ¿El número llegó como texto? Convertilo con int() o float()."),
    ("can only concatenate str", "Estás sumando texto con otra cosa. Convertí con str() para unir textos, o con float() para hacer cuentas."),
    ("unsupported operand type", "Los dos lados de la operación tienen tipos incompatibles (por ejemplo, int + str o None + float). Revisá type() de cada valor."),
    ("could not convert string to float", "Hay un texto que no representa un número (por ejemplo 'n/d' o ''). Filtralo o capturá el ValueError con try/except."),
    ("invalid literal for int", "Hay un texto que no representa un entero. Revisá el valor o usá try/except ValueError."),
    ("index out of range", "Pediste una posición que no existe. Recordá: las posiciones van de 0 a len()-1, y -1 es el último."),
    ("values to unpack", "La cantidad de variables a la izquierda no coincide con la cantidad de valores que llegan."),
    ("ZeroDivisionError", "Dividiste por cero. ¿La lista o el divisor pueden estar vacíos? Verificalo antes de dividir."),
    ("'NoneType' object", "Un valor es None. Suele pasar con una función sin return o con métodos como .sort() que devuelven None."),
    ("KeyError", "Esa clave o columna no existe. Revisá mayúsculas, tildes y espacios; con dicts podés usar .get()."),
    ("NameError", "Usás una variable que no existe (todavía). ¿Error de tipeo o la definiste más abajo?"),
    ("IndentationError", "Revisá la sangría: el contenido de if, for, def, etc. va con 4 espacios."),
    ("SyntaxError", "Error de sintaxis: revisá paréntesis, comillas, dos puntos (:) al final de if/for/def y que cada línea esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este tipo de objeto. Revisá el nombre y qué tipo tiene la variable."),
    ("MergeError", "Revisá las columnas que usás para unir (on / left_on / right_on)."),
    ("TypeError", "Algún tipo de dato no encaja con la operación. Revisá type() de las variables involucradas."),
    ("ValueError", "El tipo es correcto pero el valor no es válido para esta operación."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n\ntemperaturas = np.array([21.5, 23.0, 19.8, 25.1, 27.3, 22.4, 18.9])   # una semana, en °C\nnotas = np.array([[7, 8, 6],\n                  [9, 5, 8],\n                  [6, 7, 9],\n                  [8, 9, 10],\n                  [5, 6, 7]])                                          # 5 alumnos x 3 exámenes\nprecios = productos["precio"].to_numpy()\n\n_rng = np.random.default_rng(3)\nsensores = np.round(22 + _rng.normal(0, 1.5, size=(7, 4)), 1)   # 7 días x 4 sensores (°C)\nsensores[2, 1] = 41.0                                            # falla del sensor 1 el día 2\ndel _rng\n'
_SOL_B64 = 'eyJucC1lMDEiOiB7InNvbHV0aW9uIjogInByZWNpb3NfbGlzdGEgPSBbMTAsIDIwLCAzMF1cbnJlc3VsdGFkbyA9IG5wLmFycmF5KHByZWNpb3NfbGlzdGEpICogMiIsICJjaGVjayI6IHt9LCAiaGludHMiOiBbIk1pcsOhIGVsIGxhcmdvIGRlbCByZXN1bHRhZG8uIiwgIsK/UXXDqSBzaWduaWZpY2EgYCpgIHBhcmEgdW5hIGxpc3RhPyIsICJDb252ZXJ0w60gbGEgbGlzdGEgZW4gdW4gYXJyYXkgYW50ZXMgZGUgbXVsdGlwbGljYXIuIl0sICJleHBsYWluIjogIlBhcmEgdW5hIGxpc3RhIGRlIFB5dGhvbiwgYCogMmAgc2lnbmlmaWNhICdyZXBldGlyIGxhIGxpc3RhIGRvcyB2ZWNlcycuIFBhcmEgbXVsdGlwbGljYXIgY2FkYSBlbGVtZW50byBoYXkgcXVlIGNvbnZlcnRpcmxhIGVuIGFycmF5IChgbnAuYXJyYXkoLi4uKWApLCBkb25kZSBsYXMgb3BlcmFjaW9uZXMgc29uIGVsZW1lbnRvIGEgZWxlbWVudG8uIiwgInBzZXVkbyI6ICJyZXN1bHRhZG8g4oaQIGFycmF5KHByZWNpb3NfbGlzdGEpIMOXIDIifSwgIm5wMDEiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IG5wLmFycmF5KFs1LCAxMCwgMTUsIDIwXSkiLCAiY2hlY2siOiB7fSwgImhpbnRzIjogWyJUZW7DqXMgdW5hIGxpc3RhIGRlIHZhbG9yZXMgeSBxdWVyw6lzIHVuIGFycmF5LiIsICJIYXkgdW5hIGZ1bmNpw7NuIHF1ZSBjb252aWVydGUgbGlzdGFzIGVuIGFycmF5cy4iLCAibnAuYXJyYXkoWy4uLl0pIl0sICJleHBsYWluIjogImBucC5hcnJheWAgY29udmllcnRlIHVuYSBsaXN0YSBkZSBQeXRob24gZW4gdW4gYXJyYXkgZGUgTnVtUHkgKHRvZG9zIGRlbCBtaXNtbyB0aXBvKS4iLCAicHNldWRvIjogInJlc3VsdGFkbyDihpAgYXJyYXkgY29uIFs1LCAxMCwgMTUsIDIwXSJ9LCAibnAwMiI6IHsic29sdXRpb24iOiAicmVzdWx0YWRvID0gbnAuYXJhbmdlKDAsIDUwLCA1KSIsICJjaGVjayI6IHt9LCAiaGludHMiOiBbIkVzIHVuYSBzZWN1ZW5jaWEgcmVndWxhcjogZGUgMCBhIDQ1IGRlIDUgZW4gNS4iLCAiRXhpc3RlIHVuYSBmdW5jacOzbiBxdWUgZ2VuZXJhIHJhbmdvcyAoY29tbyByYW5nZSBwZXJvIGRldnVlbHZlIHVuIGFycmF5KS4iLCAibnAuYXJhbmdlKGluaWNpbywgZmluX2V4Y2x1aWRvLCBwYXNvKS4iXSwgImV4cGxhaW4iOiAiYG5wLmFyYW5nZShpbmljaW8sIGZpbiwgcGFzbylgIGdlbmVyYSBsYSBzZWN1ZW5jaWE7IGVsIGZpbiBubyBzZSBpbmNsdXllLCBwb3IgZXNvIHNlIHVzYSA1MC4iLCAicHNldWRvIjogInJlc3VsdGFkbyDihpAgc2VjdWVuY2lhIGRlIDAgYSA0NSBkZSA1IGVuIDUgKGZpbiBleGNsdWlkbzogNTApIn0sICJucDAzIjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSBub3Rhcy5zaGFwZSIsICJjaGVjayI6IHt9LCAiaGludHMiOiBbIlF1ZXLDqXMgc2FiZXIgY3XDoW50YXMgZmlsYXMgeSBjb2x1bW5hcyB0aWVuZSBsYSBtYXRyaXouIiwgIkVzIHVuIGF0cmlidXRvIGRlbCBhcnJheS4iLCAiTWlzbW8gbm9tYnJlIHF1ZSBlbiBwYW5kYXMuIl0sICJleHBsYWluIjogImBzaGFwZWAgZGV2dWVsdmUgbGEgZm9ybWE6ICg1LCAzKSA9IDUgYWx1bW5vcyAoZmlsYXMpIMOXIDMgZXjDoW1lbmVzIChjb2x1bW5hcykuIiwgInBzZXVkbyI6ICJyZXN1bHRhZG8g4oaQIG5vdGFzLnNoYXBlIn0sICJucDA0IjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSBucC5hcmFuZ2UoMTIpLnJlc2hhcGUoMywgNCkiLCAiY2hlY2siOiB7fSwgImhpbnRzIjogWyJTb24gMTIgbsO6bWVyb3MgZW4gMyBmaWxhcyBkZSA0LiIsICJQcmltZXJvIGdlbmVyw6EgbG9zIDEyIG7Dum1lcm9zLCBkZXNwdcOpcyBjYW1iaWFsZXMgbGEgZm9ybWEuIiwgImFyYW5nZSgxMikgKyByZXNoYXBlKGZpbGFzLCBjb2x1bW5hcykuIl0sICJleHBsYWluIjogIlNlIGdlbmVyYSB1biBhcnJheSBkZSAxMiBlbGVtZW50b3MgeSBgcmVzaGFwZSgzLCA0KWAgbG8gcmVvcmdhbml6YSBlbiAzIGZpbGFzIMOXIDQgY29sdW1uYXMgKDEyID0gMyDDlyA0KS4iLCAicHNldWRvIjogInJlc3VsdGFkbyDihpAgYXJhbmdlKDEyKSByZW9yZ2FuaXphZG8gZW4gMyDDlyA0In0sICJucC1kMDEiOiB7InNvbHV0aW9uIjogImxlY3R1cmFzX3R4dCA9IG5wLmFycmF5KFsnMjEuNScsICcyMy4wJywgJzE5LjgnLCAnMjUuMSddKVxucHJpbnQobGVjdHVyYXNfdHh0LmR0eXBlKVxuXG5yZXN1bHRhZG8gPSBsZWN0dXJhc190eHQuYXN0eXBlKGZsb2F0KS5tZWFuKCkiLCAiY2hlY2siOiB7fSwgImhpbnRzIjogWyJNaXLDoSBlbCBkdHlwZSBxdWUgaW1wcmltZTogwr9zZSBwdWVkZSBwcm9tZWRpYXIgdGV4dG8/IiwgIlByaW1lcm8gaGF5IHF1ZSBjYW1iaWFyIGVsIHRpcG8gZGVsIGFycmF5LiIsICJhc3R5cGUoZmxvYXQpIHkgZGVzcHXDqXMgbWVhbigpLiJdLCAiZXhwbGFpbiI6ICJFbCBhcnJheSBlcyBkZSB0ZXh0byAoZHR5cGUgJzxVNCcpOyBoYXkgcXVlIGNvbnZlcnRpcmxvIGNvbiBgYXN0eXBlKGZsb2F0KWAgYW50ZXMgZGUgY2FsY3VsYXIgZWwgcHJvbWVkaW8uIiwgInBzZXVkbyI6ICJudW3DqXJpY28g4oaQIGxlY3R1cmFzX3R4dCBjb252ZXJ0aWRvIGEgZmxvYXRcbnJlc3VsdGFkbyDihpAgcHJvbWVkaW8gZGUgbnVtw6lyaWNvIn0sICJucDA1IjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSBub3Rhc1s6LCAxXSIsICJjaGVjayI6IHt9LCAiaGludHMiOiBbIlF1ZXLDqXMgdW5hIGNvbHVtbmEgY29tcGxldGEgKHVuIGV4YW1lbiBwYXJhIHRvZG9zIGxvcyBhbHVtbm9zKS4iLCAiRW4gMkQgc2UgaW5kZXhhIGBbZmlsYXMsIGNvbHVtbmFzXWAuIiwgImA6YCBzaWduaWZpY2EgJ3RvZGFzIGxhcyBmaWxhcyc7IGVsIHNlZ3VuZG8gZXhhbWVuIGVzIGxhIGNvbHVtbmEgMS4iXSwgImV4cGxhaW4iOiAiYG5vdGFzWzosIDFdYCB0b21hIHRvZGFzIGxhcyBmaWxhcyAoYDpgKSBkZSBsYSBjb2x1bW5hIDEgKHNlZ3VuZG8gZXhhbWVuKS4iLCAicHNldWRvIjogInJlc3VsdGFkbyDihpAgbm90YXNbdG9kYXMgbGFzIGZpbGFzLCBjb2x1bW5hIDFdIn0sICJucDA2IjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSBub3Rhc1s6MywgMTpdIiwgImNoZWNrIjoge30sICJoaW50cyI6IFsiRXMgdW4gYmxvcXVlOiBhbGd1bmFzIGZpbGFzIHkgYWxndW5hcyBjb2x1bW5hcy4iLCAiUHJpbWVyb3MgMyBhbHVtbm9zID0gZmlsYXMgMCBhIDI7IMO6bHRpbW9zIDIgZXjDoW1lbmVzID0gY29sdW1uYXMgMSB5IDIuIiwgIlNsaWNpbmcgZW4gY2FkYSBkaW1lbnNpw7NuOiBgWzozLCAxOl1gLiJdLCAiZXhwbGFpbiI6ICJTZSByZWNvcnRhIGVuIGFtYmFzIGRpbWVuc2lvbmVzOiBmaWxhcyBoYXN0YSBsYSAzIChleGNsdWlkYSkgeSBjb2x1bW5hcyBkZXNkZSBsYSAxLiIsICJwc2V1ZG8iOiAicmVzdWx0YWRvIOKGkCBub3Rhc1tmaWxhcyAwLTIsIGNvbHVtbmFzIDEtMl0ifSwgIm5wMDciOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IHRlbXBlcmF0dXJhc1t0ZW1wZXJhdHVyYXMgPiAyMl0iLCAiY2hlY2siOiB7fSwgImhpbnRzIjogWyJQcmltZXJvIHByZWd1bnTDoSAobcOhc2NhcmEpLCBkZXNwdcOpcyBmaWx0csOhLiIsICJgdGVtcGVyYXR1cmFzID4gMjJgIGRhIFRydWUvRmFsc2UgcG9yIGVsZW1lbnRvLiIsICJFc2EgbcOhc2NhcmEgZW50cmUgY29yY2hldGVzIHNlIHF1ZWRhIGNvbiBsb3MgVHJ1ZS4iXSwgImV4cGxhaW4iOiAiSWd1YWwgcXVlIGVuIHBhbmRhczogbGEgbcOhc2NhcmEgYm9vbGVhbmEgZW50cmUgY29yY2hldGVzIGRldnVlbHZlIHNvbG8gbG9zIGVsZW1lbnRvcyBxdWUgY3VtcGxlbi4iLCAicHNldWRvIjogInJlc3VsdGFkbyDihpAgdGVtcGVyYXR1cmFzW3RlbXBlcmF0dXJhcyA+IDIyXSJ9LCAibnAwOCI6IHsic29sdXRpb24iOiAicmVzdWx0YWRvID0gdGVtcGVyYXR1cmFzICogOSAvIDUgKyAzMiIsICJjaGVjayI6IHt9LCAiaGludHMiOiBbIkxhIGbDs3JtdWxhIGVzIGxhIG1pc21hIHBhcmEgY2FkYSB0ZW1wZXJhdHVyYS4iLCAiQ29uIGFycmF5cyBubyBoYWNlIGZhbHRhIHVuIGZvci4iLCAiRXNjcmliw60gbGEgZsOzcm11bGEgdXNhbmRvIGVsIGFycmF5IGNvbW8gc2kgZnVlcmEgdW4gbsO6bWVyby4iXSwgImV4cGxhaW4iOiAiTGEgZsOzcm11bGEgc2UgYXBsaWNhIGEgdG9kbyBlbCBhcnJheSBhIGxhIHZleiAob3BlcmFjacOzbiB2ZWN0b3JpemFkYSkuIiwgInBzZXVkbyI6ICJyZXN1bHRhZG8g4oaQIHRlbXBlcmF0dXJhcyDDlyA5LzUgKyAzMiJ9LCAibnAxMCI6IHsic29sdXRpb24iOiAiYm9udXMgPSBucC5hcnJheShbMSwgMCwgMC41XSlcbnJlc3VsdGFkbyA9IG5wLm1pbmltdW0obm90YXMgKyBib251cywgMTApIiwgImNoZWNrIjoge30sICJoaW50cyI6IFsiRWwgYm9udXMgZXMgdW5vIHBvciBleGFtZW4gKDMgdmFsb3JlcykgeSBsYSBtYXRyaXogZXMgNSDDlyAzLiIsICJQb3IgYnJvYWRjYXN0aW5nLCB1biB2ZWN0b3IgZGUgMyBzZSBzdW1hIGEgY2FkYSBmaWxhLiIsICJEZXNwdcOpcywgbnAubWluaW11bSguLi4sIDEwKSBwb25lIGVsIHRvcGUuIl0sICJleHBsYWluIjogIkVsIHZlY3RvciBkZSBib251cyAoMywpIHNlIHN1bWEgYSBjYWRhIGZpbGEgZGUgbGEgbWF0cml6ICg1LCAzKSBwb3IgYnJvYWRjYXN0aW5nOyBgbnAubWluaW11bWAgbGltaXRhIGNhZGEgbm90YSBhIDEwLiIsICJwc2V1ZG8iOiAiYm9udXMg4oaQIFsxLCAwLCAwLjVdXG5yZXN1bHRhZG8g4oaQIG3DrW5pbW8obm90YXMgKyBib251cywgMTApIn0sICJucC1lMDIiOiB7InNvbHV0aW9uIjogImJvbnVzX2FsdW1ubyA9IG5wLmFycmF5KFsxLCAwLCAwLjUsIDAsIDFdKVxucmVzdWx0YWRvID0gbnAubWluaW11bShub3RhcyArIGJvbnVzX2FsdW1uby5yZXNoYXBlKDUsIDEpLCAxMCkiLCAiY2hlY2siOiB7fSwgImhpbnRzIjogWyJMZcOpIGVsIGVycm9yOiDCv3F1w6kgc2hhcGVzIG1lbmNpb25hPyIsICLCv0NvbnRyYSBxdcOpIGRpbWVuc2nDs24gZGUgbGEgbWF0cml6IHNlIGFsaW5lYSB1biB2ZWN0b3IgZGUgbGFyZ28gNT8iLCAiUHJvYsOhIGRhcmxlIGFsIGJvbnVzIGxhIGZvcm1hICg1LCAxKS4iXSwgImV4cGxhaW4iOiAiQnJvYWRjYXN0aW5nIGNvbXBhcmEgbGFzIGZvcm1hcyAqKmRlc2RlIGxhIGRlcmVjaGEqKjogKDUsIDMpIGNvbnRyYSAoNSwpIOKGkiBjb21wYXJhIDMgY29uIDUgeSBmYWxsYS4gQ29udmlydGllbmRvIGVsIGJvbnVzIGVuIGNvbHVtbmEgYCg1LCAxKWAsIGxhIGNvbXBhcmFjacOzbiBlcyAzIHZzIDEgKG9rKSB5IDUgdnMgNSAob2spOiBjYWRhIGFsdW1ubyBzdW1hIHN1IGJvbnVzIGVuIHN1cyAzIGV4w6FtZW5lcy4iLCAicHNldWRvIjogInJlc3VsdGFkbyDihpAgbcOtbmltbyhub3RhcyArIGJvbnVzIGNvbW8gY29sdW1uYSAoNSwxKSwgMTApIn0sICJucDExIjogeyJzb2x1dGlvbiI6ICJwcm9tZWRpb3MgPSBub3Rhcy5tZWFuKGF4aXM9MSlcbnJlc3VsdGFkbyA9IG5wLndoZXJlKHByb21lZGlvcyA+PSA3LCAnYXByb2JhZG8nLCAncmVjdXBlcmEnKSIsICJjaGVjayI6IHt9LCAiaGludHMiOiBbIlByaW1lcm8gbmVjZXNpdMOhcyBlbCBwcm9tZWRpbyBkZSBjYWRhIGFsdW1uby4iLCAiRGVzcHXDqXMsIHVuYSBkZWNpc2nDs24gcG9yIGFsdW1ubzogJ2Fwcm9iYWRvJyBvICdyZWN1cGVyYScuIiwgIm5wLndoZXJlKGNvbmRpY2nDs24sIHZhbG9yX3NpLCB2YWxvcl9ubykuIl0sICJleHBsYWluIjogIlNlIGNhbGN1bGEgZWwgcHJvbWVkaW8gcG9yIGZpbGEgeSBgbnAud2hlcmVgIGVsaWdlIGxhIGV0aXF1ZXRhIHBhcmEgY2FkYSBhbHVtbm8gc2Vnw7puIGxhIGNvbmRpY2nDs24uIiwgInBzZXVkbyI6ICJwcm9tZWRpb3Mg4oaQIG5vdGFzLm1lYW4oYXhpcz0xKVxucmVzdWx0YWRvIOKGkCB3aGVyZShwcm9tZWRpb3Mg4omlIDcsICdhcHJvYmFkbycsICdyZWN1cGVyYScpIn0sICJucC1mMDEiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IChwcmVjaW9zIC0gcHJlY2lvcy5taW4oKSkgLyAocHJlY2lvcy5tYXgoKSAtIHByZWNpb3MubWluKCkpIiwgImNoZWNrIjoge30sICJoaW50cyI6IFsiRWplY3V0YWxvIHkgbWlyw6EgZWwgbcOtbmltbyB5IG3DoXhpbW8gZGVsIHJlc3VsdGFkby4iLCAiUHJvYsOhIGNvbiB1biBhcnJheSBkZSB2YWxvcmVzIGlndWFsZXMuIiwgIlBlbnPDoSBlbiBxdcOpIG1vZGVsb3MgaW1wb3J0YSBsYSBlc2NhbGEuIl0sICJleHBsYWluIjogIlNlIHJlc3RhIGVsIG3DrW5pbW8gKGVsIG3DoXMgYmFyYXRvIHF1ZWRhIGVuIDApIHkgc2UgZGl2aWRlIHBvciBlbCByYW5nbyAoZWwgbcOhcyBjYXJvIHF1ZWRhIGVuIDEpLiBDYXNvIGJvcmRlOiByYW5nbyAwLiIsICJwc2V1ZG8iOiAiIn0sICJucDA5IjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSBub3Rhcy5tZWFuKGF4aXM9MSkiLCAiY2hlY2siOiB7fSwgImhpbnRzIjogWyJRdWVyw6lzIHVuIHZhbG9yIHBvciBhbHVtbm8gKHBvciBmaWxhKS4iLCAibWVhbiBhY2VwdGEgZWwgcGFyw6FtZXRybyBheGlzLiIsICJheGlzPTEgY29sYXBzYSBsYXMgY29sdW1uYXMuIl0sICJleHBsYWluIjogImBtZWFuKGF4aXM9MSlgIHByb21lZGlhIGEgbG8gbGFyZ28gZGUgbGFzIGNvbHVtbmFzOiB1biBwcm9tZWRpbyBwb3IgZmlsYSAoYWx1bW5vKS4iLCAicHNldWRvIjogInJlc3VsdGFkbyDihpAgcHJvbWVkaW8gZGUgY2FkYSBmaWxhIGRlIG5vdGFzIn0sICJucC1nMDEiOiB7InNvbHV0aW9uIjogInByb21lZGlvc19leGFtZW4gPSBub3Rhcy5tZWFuKGF4aXM9MClcbnJlc3VsdGFkbyA9IHByb21lZGlvc19leGFtZW4uYXJnbWF4KCkiLCAiY2hlY2siOiB7fSwgImhpbnRzIjogWyLCv1F1w6kgZWplIGRhIHVuIHZhbG9yIHBvciBjb2x1bW5hPyIsICJUZW7DqXMgMyBwcm9tZWRpb3M7IHF1ZXLDqXMgbGEgKipwb3NpY2nDs24qKiBkZWwgbWF5b3IsIG5vIHN1IHZhbG9yLiIsICJFeGlzdGUgdW4gbcOpdG9kbyBxdWUgZGV2dWVsdmUgbGEgcG9zaWNpw7NuIGRlbCBtw6F4aW1vOiBhcmdtYXguIl0sICJleHBsYWluIjogIkNvbiBheGlzPTAgc2Ugb2J0aWVuZSB1biBwcm9tZWRpbyBwb3IgZXhhbWVuOyBgYXJnbWF4KClgIGRldnVlbHZlIGxhIHBvc2ljacOzbiBkZWwgbWF5b3IgKGVsIGV4YW1lbiBtw6FzIGbDoWNpbCkuIiwgInBzZXVkbyI6ICJwcm9tZWRpb3Mg4oaQIG5vdGFzLm1lYW4oYXhpcz0wKVxucmVzdWx0YWRvIOKGkCBwb3NpY2nDs24gZGVsIG3DoXhpbW8gZGUgcHJvbWVkaW9zIn0sICJucDEzIjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSAodGVtcGVyYXR1cmFzIC0gdGVtcGVyYXR1cmFzLm1lYW4oKSkgLyB0ZW1wZXJhdHVyYXMuc3RkKCkiLCAiY2hlY2siOiB7fSwgImhpbnRzIjogWyJFc3RhbmRhcml6YXIgPSBjZW50cmFyIGVuIDAgeSBtZWRpciBlbiBkZXN2w61vcy4iLCAiTmVjZXNpdMOhcyBsYSBtZWRpYSB5IGVsIGRlc3bDrW8gZXN0w6FuZGFyIGRlbCBhcnJheS4iLCAiKHgg4oiSIG1lZGlhKSAvIGRlc3bDrW8sIGFwbGljYWRvIGEgdG9kbyBlbCBhcnJheS4iXSwgImV4cGxhaW4iOiAiU2UgcmVzdGEgbGEgbWVkaWEgKGNlbnRyYXIpIHkgc2UgZGl2aWRlIHBvciBlbCBkZXN2w61vIGVzdMOhbmRhciAoZXNjYWxhcikuIENhZGEgdmFsb3IgcXVlZGEgZXhwcmVzYWRvIGVuICdjdcOhbnRvcyBkZXN2w61vcyBzZSBhbGVqYSBkZSBsYSBtZWRpYScuIiwgInBzZXVkbyI6ICJyZXN1bHRhZG8g4oaQICh0ZW1wZXJhdHVyYXMg4oiSIG1lZGlhKSAvIGRlc3bDrW8ifSwgIm5wMTQiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IG5wLnBlcmNlbnRpbGUocHJlY2lvcywgNzUpIiwgImNoZWNrIjoge30sICJoaW50cyI6IFsiRWwgcGVyY2VudGlsIDc1IGVzIGVsIHZhbG9yIHF1ZSBkZWphIHBvciBkZWJham8gYWwgNzUlIGRlIGxvcyBkYXRvcy4iLCAiTnVtUHkgdGllbmUgdW5hIGZ1bmNpw7NuIGRlIHBlcmNlbnRpbGVzLiIsICJucC5wZXJjZW50aWxlKGRhdG9zLCA3NSkuIl0sICJleHBsYWluIjogImBucC5wZXJjZW50aWxlYCBjYWxjdWxhIGVsIHZhbG9yIHBvciBkZWJham8gZGVsIGN1YWwgcXVlZGEgZWwgNzUlIGRlIGxvcyBwcmVjaW9zLiIsICJwc2V1ZG8iOiAicmVzdWx0YWRvIOKGkCBwZXJjZW50aWwgNzUgZGUgcHJlY2lvcyJ9LCAibnAtZDAyIjogeyJzb2x1dGlvbiI6ICJ6ID0gKHRlbXBlcmF0dXJhcyAtIHRlbXBlcmF0dXJhcy5tZWFuKCkpIC8gdGVtcGVyYXR1cmFzLnN0ZCgpXG5yZXN1bHRhZG8gPSB0ZW1wZXJhdHVyYXNbbnAuYWJzKHopID4gMS41XSIsICJjaGVjayI6IHt9LCAiaGludHMiOiBbIlNlZ3XDrSB0dSBwbGFuIHBhc28gYSBwYXNvLiIsICJDYWxjdWzDoSB6IHBhcmEgdG9kbyBlbCBhcnJheSBlbiB1bmEgbMOtbmVhLiIsICJucC5hYnMoeikgPiAxLjUgY29tbyBtw6FzY2FyYS4iXSwgImV4cGxhaW4iOiAiU2UgY2FsY3VsYSBlbCB6LXNjb3JlIHZlY3Rvcml6YWRvIHkgc2UgZmlsdHJhIGNvbiBsYSBtw6FzY2FyYSB8enwgPiAxLjUuIiwgInBzZXVkbyI6ICJ6IOKGkCAodCDiiJIgbWVkaWEpIC8gZGVzdsOtb1xucmVzdWx0YWRvIOKGkCB0W3x6fCA+IDEuNV0ifSwgIm5wMTIiOiB7InNvbHV0aW9uIjogInJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZygwKVxucmVzdWx0YWRvID0gcm5nLmludGVnZXJzKDEsIDcsIHNpemU9MTApIiwgImNoZWNrIjoge30sICJoaW50cyI6IFsiVGllbmUgcXVlIHNlciByZXByb2R1Y2libGU6IMK/cXXDqSBuZWNlc2l0w6FzIGZpamFyPyIsICJDcmXDoSB1biBnZW5lcmFkb3IgY29uIHNlbWlsbGEgMC4iLCAicm5nLmludGVnZXJzKDEsIDcsIHNpemU9MTApOiBlbCA3IG5vIHNlIGluY2x1eWUuIl0sICJleHBsYWluIjogIkNvbiB1biBnZW5lcmFkb3IgY29uIHNlbWlsbGEgZmlqYSwgbGEgc2VjdWVuY2lhICdhbGVhdG9yaWEnIGVzIHNpZW1wcmUgbGEgbWlzbWEuIGBpbnRlZ2VycygxLCA3KWAgZGEgZW50ZXJvcyBkZWwgMSBhbCA2LiIsICJwc2V1ZG8iOiAicm5nIOKGkCBnZW5lcmFkb3IgY29uIHNlbWlsbGEgMFxucmVzdWx0YWRvIOKGkCAxMCBlbnRlcm9zIGVudHJlIDEgeSA2In0sICJucDE1IjogeyJzb2x1dGlvbiI6ICJwZXNvcyA9IG5wLmFycmF5KFswLjMsIDAuMywgMC40XSlcbnJlc3VsdGFkbyA9IG5vdGFzIEAgcGVzb3MiLCAiY2hlY2siOiB7fSwgImhpbnRzIjogWyJDYWRhIGFsdW1ubyB0aWVuZSAzIG5vdGFzIHkgY2FkYSBleGFtZW4gdW4gcGVzbzogZXMgdW4gcHJvbWVkaW8gcG9uZGVyYWRvLiIsICJNdWx0aXBsaWNhciBjYWRhIG5vdGEgcG9yIHN1IHBlc28geSBzdW1hciBwb3IgZmlsYeKApiIsICLigKZlcyBleGFjdGFtZW50ZSB1biBwcm9kdWN0byBtYXRyaWNpYWw6IG5vdGFzIEAgcGVzb3MuIl0sICJleHBsYWluIjogIkVsIHByb2R1Y3RvIG1hdHJpY2lhbCBgbm90YXMgQCBwZXNvc2AgbXVsdGlwbGljYSBjYWRhIGZpbGEgcG9yIGVsIHZlY3RvciBkZSBwZXNvcyB5IHN1bWE6IGVsIHByb21lZGlvIHBvbmRlcmFkbyBkZSBjYWRhIGFsdW1uby4iLCAicHNldWRvIjogInBlc29zIOKGkCBbMC4zLCAwLjMsIDAuNF1cbnJlc3VsdGFkbyDihpAgbm90YXMgQCBwZXNvcyJ9LCAibnAtZDAzIjogeyJzb2x1dGlvbiI6ICJyZXN1bHRhZG8gPSBucC5jb3JyY29lZih2ZW50YXNfZGlhcmlhc1sndmlzaXRhcyddLCB2ZW50YXNfZGlhcmlhc1sndmVudGFzJ10pWzAsIDFdIiwgImNoZWNrIjoge30sICJoaW50cyI6IFsiTGEgY29ycmVsYWNpw7NuIG1pZGUgc2kgZG9zIHZhcmlhYmxlcyBjcmVjZW4ganVudGFzLiIsICJOdW1QeSB0aWVuZSB1bmEgZnVuY2nDs24gcXVlIGRldnVlbHZlIGxhIG1hdHJpeiBkZSBjb3JyZWxhY2nDs24uIiwgIm5wLmNvcnJjb2VmKGEsIGIpIGRldnVlbHZlIHVuYSBtYXRyaXogMsOXMjsgZWwgdmFsb3IgcXVlIGJ1c2PDoXMgZXN0w6EgZnVlcmEgZGUgbGEgZGlhZ29uYWw6IFswLCAxXS4iXSwgImV4cGxhaW4iOiAiYG5wLmNvcnJjb2VmYCBhY2VwdGEgbGFzIGNvbHVtbmFzIGRlIHBhbmRhcyBkaXJlY3RhbWVudGUgeSBkZXZ1ZWx2ZSBsYSBtYXRyaXogZGUgY29ycmVsYWNpw7NuIDLDlzI7IGVsIGVsZW1lbnRvIFswLCAxXSBlcyBsYSBjb3JyZWxhY2nDs24gZW50cmUgYW1iYXMgdmFyaWFibGVzLiIsICJwc2V1ZG8iOiAibWF0cml6IOKGkCBjb3JyY29lZih2aXNpdGFzLCB2ZW50YXMpXG5yZXN1bHRhZG8g4oaQIG1hdHJpelswLCAxXSJ9LCAibnAtcDIiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IG5wLnJvdW5kKHNlbnNvcmVzLm1lYW4oYXhpcz0wKSwgMikiLCAiY2hlY2siOiB7fSwgImhpbnRzIjogWyJVbiB2YWxvciBwb3Igc2Vuc29yOiDCv3BvciBmaWxhIG8gcG9yIGNvbHVtbmE/IiwgIkxvcyBzZW5zb3JlcyBzb24gbGFzIGNvbHVtbmFzLiIsICJtZWFuKGF4aXM9MCkgeSBkZXNwdcOpcyBucC5yb3VuZCguLi4sIDIpLiJdLCAiZXhwbGFpbiI6ICJDb24gYXhpcz0wIHNlIGNvbGFwc2FuIGxvcyBkw61hcyB5IHF1ZWRhIHVuIHByb21lZGlvIHBvciBzZW5zb3IuIEVsIHNlbnNvciAxIHNhbGUgbcOhcyBhbHRvIHBvciBsYSBsZWN0dXJhIGRlIDQxIMKwQy4iLCAicHNldWRvIjogInJlc3VsdGFkbyDihpAgcmVkb25kZWFyKG1lZGlhIHBvciBjb2x1bW5hLCAyKSJ9LCAibnAtcDMiOiB7InNvbHV0aW9uIjogImxpbWl0ZSA9IHNlbnNvcmVzLm1lYW4oKSArIDMgKiBzZW5zb3Jlcy5zdGQoKVxucmVzdWx0YWRvID0gbnAuYXJnd2hlcmUoc2Vuc29yZXMgPiBsaW1pdGUpIiwgImNoZWNrIjoge30sICJoaW50cyI6IFsiUHJpbWVybyBjYWxjdWzDoSBlbCBsw61taXRlIGNvbiBlc3RhZMOtc3RpY2FzIGRlIHRvZGEgbGEgbWF0cml6LiIsICJMYSBjb25kaWNpw7NuIGBzZW5zb3JlcyA+IGxpbWl0ZWAgZXMgdW5hIG1hdHJpeiBkZSBUcnVlL0ZhbHNlLiIsICJucC5hcmd3aGVyZShjb25kaWNpw7NuKSBkZXZ1ZWx2ZSB1bmEgZmlsYSBbZMOtYSwgc2Vuc29yXSBwb3IgY2FkYSBUcnVlLiJdLCAiZXhwbGFpbiI6ICJTZSBjYWxjdWxhIHVuIHVtYnJhbCBnbG9iYWwgeSBgbnAuYXJnd2hlcmVgIGRldnVlbHZlIGxhcyBjb29yZGVuYWRhcyBkZSBsb3MgdmFsb3JlcyBxdWUgbG8gc3VwZXJhbjogZMOtYSAyLCBzZW5zb3IgMS4iLCAicHNldWRvIjogImzDrW1pdGUg4oaQIG1lZGlhICsgMyDDlyBkZXN2w61vXG5yZXN1bHRhZG8g4oaQIHBvc2ljaW9uZXMgZG9uZGUgc2Vuc29yZXMgPiBsw61taXRlIn0sICJucC1wNCI6IHsic29sdXRpb24iOiAicmVzdWx0YWRvID0gbnAucm91bmQoc2Vuc29yZXMubWVhbihheGlzPTApLCAyKSIsICJjaGVjayI6IHt9LCAiaGludHMiOiBbIsK/Q3XDoW50b3Mgc2Vuc29yZXMgaGF5IHkgY3XDoW50b3MgdmFsb3JlcyBzYWxlbj8iLCAiwr9RdcOpIHJlcHJlc2VudGEgY2FkYSBmaWxhPyIsICLCv1F1w6kgYXhpcyBkZWphIHVuIHZhbG9yIHBvciBjb2x1bW5hPyJdLCAiZXhwbGFpbiI6ICJheGlzPTEgY29sYXBzYSBsYXMgY29sdW1uYXMgeSBkZWphIHVuIHZhbG9yIHBvciAqKmZpbGEqKiAoZMOtYSk6IDcgdmFsb3Jlcy4gUGFyYSB1biB2YWxvciBwb3IgKipzZW5zb3IqKiAoY29sdW1uYSkgaGF5IHF1ZSBjb2xhcHNhciBsYXMgZmlsYXM6IGF4aXM9MC4gVmVyaWZpY2FyIGVsIGxhcmdvIGRlbCByZXN1bHRhZG8gZGV0ZWN0YSBlc3RlIGVycm9yIGFsIGluc3RhbnRlLiIsICJwc2V1ZG8iOiAicmVzdWx0YWRvIOKGkCByZWRvbmRlYXIobWVkaWEgY29uIGF4aXM9MCwgMikifX0='
_TABLAS = ['temperaturas', 'notas', 'precios', 'sensores', 'ventas_diarias']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todos los datos limpios (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())
    pq_prepare()

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    def esperado():
        exp_ns = {}
        exec(_SETUP, exp_ns)
        exec(info["solution"], exp_ns)
        return exp_ns
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, esperado, info["check"])
    from IPython.display import HTML, display
    import html as _h
    if ok:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">✅ Verificado. Ahora explicá con tus palabras por qué funciona '
                     'y probá una variante.</div>'))
    else:
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_pista(ej, nivel=1):
    """Pistas progresivas (1-3). Pedí la siguiente solo si la anterior no alcanzó."""
    pistas = _SOL[ej]["hints"]
    print(f"Pista {nivel}/{len(pistas)}: {pistas[min(nivel, len(pistas)) - 1]}")

def ver_solucion(ej):
    """Último recurso: primero la idea, después el pseudocódigo y al final el código."""
    info = _SOL[ej]
    print("💡 Idea:\n" + info["explain"] + "\n")
    print("🧭 Pseudocódigo:\n" + info["pseudo"] + "\n")
    print("🧑‍💻 Código:\n" + info["solution"])

reiniciar_datos()
print("✅ Datos cargados:", ", ".join(_TABLAS))


---
# 📌 ¿Cuándo necesito NumPy?

## 1. Duplicar precios
`Debugging` · `Muy fácil`

Querías duplicar cada precio, pero el resultado tiene el doble de elementos. Corregilo usando NumPy.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

precios_lista = [10, 20, 30]
resultado = precios_lista * 2

comprobar("np-e01")

---
# 📌 Arrays: forma, dimensiones y tipo

## 2. Tu primer array
`Código` · `Muy fácil`

Creá un array de NumPy con los valores `5, 10, 15, 20` y guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

# numpy ya está importado como np
resultado = 

comprobar("np01")

## 3. Rangos con arange y linspace
`Código` · `Fácil`

Generá el array `0, 5, 10, …, 45` sin escribirlo a mano. Guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np02")

## 4. Dimensiones y forma
`Código` · `Muy fácil`

¿Qué forma tiene la matriz `notas` (cuántos alumnos y cuántos exámenes)? Guardala en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

print(notas)
resultado = 

comprobar("np03")

## 5. Cambiar de forma
`Código` · `Medio`

Creá una matriz de **3 filas × 4 columnas** con los números del **0 al 11**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np04")

## 6. Lecturas que llegaron como texto
`Código` · `Fácil`

Las lecturas de un sensor llegaron como array de **texto**. Calculá su **promedio** y guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

lecturas_txt = np.array(['21.5', '23.0', '19.8', '25.1'])
print(lecturas_txt.dtype)

resultado = 

comprobar("np-d01")

---
# 📌 Indexar, recortar y filtrar

## 7. Acceder a elementos
`Código` · `Medio`

Guardá en `resultado` las notas del **segundo examen** de todos los alumnos.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np05")

## 8. Recortar submatrices
`Código` · `Medio`

Guardá las notas de los **3 primeros alumnos** en los **2 últimos exámenes**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np06")

## 9. Filtrar con máscaras
`Código` · `Medio`

¿Qué días hizo **más de 22 grados**? Guardá esas temperaturas en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np07")

---
# 📌 Cálculo vectorizado y broadcasting

## 10. Fórmulas sobre arrays
`Código` · `Fácil`

Convertí `temperaturas` de Celsius a **Fahrenheit** (`F = C × 9/5 + 32`).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np08")

## 11. Broadcasting
`Código` · `Difícil`

Sumá a `notas` un bonus por examen `[1, 0, 0.5]` y limitá las notas a un **máximo de 10**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

bonus = np.array([1, 0, 0.5])
resultado = 

comprobar("np10")

## 12. El bonus por alumno que no encaja
`Debugging` · `Medio`

Ahora el bonus es **por alumno** (5 valores, uno por fila). La suma falla. Corregila para que cada alumno reciba su bonus en todos sus exámenes (con tope en 10).

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

bonus_alumno = np.array([1, 0, 0.5, 0, 1])
resultado = np.minimum(notas + bonus_alumno, 10)

comprobar("np-e02")

## 13. Condicionales con np.where
`Código` · `Medio`

Con el promedio de cada alumno, armá un array que diga **"aprobado"** si el promedio es **≥ 7** y **"recupera"** si no.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

promedios = notas.mean(axis=1)
resultado = 

comprobar("np11")

## 14. La IA normaliza precios
`IA copiloto` · `Medio`

Pediste a una IA 'llevar los precios a una escala de 0 a 1' y te dio este código (normalización min-max).

> 🤖 Prompt usado:
> Tengo un array de precios y quiero escalarlos entre 0 y 1 (el más barato 0, el más caro 1). Mostrame el código y explicá la fórmula y sus casos borde.

Antes de ejecutarlo, explicá en comentarios qué hace cada línea.

In [ ]:
reiniciar_datos()  # datos limpios

resultado = (precios - precios.min()) / (precios.max() - precios.min())

comprobar("np-f01")

---
# 📌 Estadística y ejes

## 15. Agregar por filas o columnas
`Código` · `Difícil`

Calculá el **promedio de cada alumno** (cada fila de `notas`). Guardá el array de 5 promedios.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np09")

## 16. Transferencia: el examen más fácil
`Transferencia` · `Medio`

Calculá el **promedio de cada examen** y guardá en `resultado` el **número de columna** (0, 1 o 2) del examen con el promedio más alto.

> 🔁 **Qué cambia:** Antes resumiste **por alumno** (filas). Ahora resumís **por examen** (columnas) para ver cuál fue el más fácil.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np-g01")

## 17. Estandarizar datos
`Código` · `Difícil`

Estandarizá `temperaturas`: restale su media y dividí por su desvío estándar (`.std()`).

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np13")

## 18. Percentiles
`Código` · `Medio`

¿Qué precio deja por debajo al 75% de los productos? Calculá el **percentil 75** de `precios`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np14")

## 19. Implementar: temperaturas anómalas
`Código` · `Medio`

Implementá el plan: guardá en `resultado` las temperaturas de `temperaturas` cuyo z-score en valor absoluto sea **mayor a 1.5**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np-d02")

---
# 📌 NumPy en la práctica (y su relación con pandas)

## 20. Números aleatorios reproducibles
`Código` · `Medio`

Simulá **10 tiradas de un dado** (enteros del 1 al 6) de forma **reproducible**, con un generador de semilla **0**.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

rng = 
resultado = 

comprobar("np12")

## 21. Álgebra de matrices
`Código` · `Desafío`

Los exámenes pesan distinto: **30%, 30% y 40%**. Calculá la **nota final ponderada** de cada alumno.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

pesos = np.array([0.3, 0.3, 0.4])
resultado = 

comprobar("np15")

## 22. ¿Visitas y ventas se mueven juntas?
`Código` · `Medio`

Usando NumPy sobre las columnas de `ventas_diarias`, calculá el **coeficiente de correlación** entre `visitas` y `ventas` (un número entre -1 y 1). Guardalo en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np-d03")

---
# 📌 Proyecto: estadísticas de una red de sensores

## 23. Fase 2 · Promedio por sensor
`Código` · `Medio`

Calculá la **temperatura promedio de cada sensor**, redondeada a 2 decimales. Guardá el array (4 valores) en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

print(sensores)

resultado = 

comprobar("np-p2")

## 24. Fase 3 · Encontrar la falla
`Código` · `Difícil`

Encontrá las posiciones `(día, sensor)` de las lecturas **mayores a la media global + 3 desvíos estándar** (globales). Guardá el resultado de `np.argwhere` en `resultado`.

In [ ]:
reiniciar_datos()  # datos limpios

# Mi plan (con mis palabras, antes de programar):
# 

resultado = 

comprobar("np-p3")

## 25. Fase 4 · El eje equivocado
`Debugging` · `Medio`

Un compañero calculó el promedio por sensor, pero obtiene **7 valores** en lugar de 4. Corregilo.

> 🐞 El código tiene un error: ejecutalo, leé el mensaje, explicá la causa en un comentario y corregilo.

In [ ]:
reiniciar_datos()  # datos limpios

resultado = np.round(sensores.mean(axis=1), 2)

comprobar("np-p4")

---
# 🏁 Fin del notebook
Volvé a la [plataforma](https://rosalesluciano.github.io/datalab/) para los ejercicios de reconocimiento, razonamiento y el modo **¿Sé esto?**.